# Week 3 Step 3: Validate untrusted SQL

Gemini's SQL is untrusted input. I never run it raw. `validate_sql` parses it with sqlglot and then:

1. Allows only one statement
2. Allows only SELECT (WITH/CTE is still a SELECT)
3. Blocks INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, CREATE
4. Checks table and column names against my schema
5. Adds `LIMIT 1000` if the query has no cap

Every attempt, including blocked ones, is appended to `logs/copilot_sql.jsonl`. Queries that pass will run as a read-only Postgres role `copilot_ro`, with `statement_timeout = 10s`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.guard_tests import run_guard_tests
from src.llm.validate_sql import validate_sql

results = pd.DataFrame(run_guard_tests())
display(results)
print("all passed:" , bool(results["pass"].all()))

Parser tests are not enough. I also create a read-only login `copilot_ro` so even a missed DROP would be refused by Postgres.

Add this to `.env` (not in git):

`COPILOT_RO_PASSWORD=choose-a-readonly-password`

Then run the next cell. It creates or updates the role, grants SELECT only, and checks that INSERT is denied.

In [ ]:
import os

from dotenv import load_dotenv
from sqlalchemy import text

load_dotenv(PROJECT_ROOT / ".env")

from src.llm.db import copilot_engine, ensure_copilot_role

if not os.getenv("COPILOT_RO_PASSWORD") and not os.getenv("COPILOT_DB_URL"):
    print("Add COPILOT_RO_PASSWORD to .env, restart the kernel, and rerun this cell.")
else:
    role = ensure_copilot_role()
    print("role ready:", role)
    engine = copilot_engine()
    with engine.connect() as conn:
        who = conn.execute(text("SELECT current_user, current_setting('statement_timeout')")).one()
        n = conn.execute(text("SELECT COUNT(*) FROM customers")).scalar()
        print("connected as:", who[0], "timeout:", who[1], "customers:", n)
        try:
            conn.execute(text("INSERT INTO customers (customer_id) VALUES ('HACK')"))
            print("ERROR: INSERT was allowed")
        except Exception as exc:
            print("INSERT blocked by Postgres:", type(exc).__name__)

In [ ]:
from src.llm.query_log import LOG_PATH

print("log file:", LOG_PATH)
if LOG_PATH.exists():
    lines = LOG_PATH.read_text(encoding="utf-8").strip().splitlines()
    print("entries:", len(lines))
    print(lines[-1] if lines else "empty")